# **Lab:** Security in Computer Networks

**Student name:** write your name here

### **Topics**
- Confidentiality
- Public Key Cryptography (RSA)
- Digital Signatures
- TLS Handshake
- Attack Simulation (CTF)

In [ ]:
# Install cryptography library (only first time)
# This library provides tools for encryption, decryption, signatures, etc.
%pip install cryptography

In [ ]:
# Import RSA module for asymmetric cryptography
from cryptography.hazmat.primitives.asymmetric import rsa

# Import serialization tools (used for saving/loading keys, not used deeply here)
from cryptography.hazmat.primitives import serialization

# Import padding schemes (needed for secure encryption)
from cryptography.hazmat.primitives.asymmetric import padding

# Import hashing algorithms (SHA256, etc.)
from cryptography.hazmat.primitives import hashes

## **🔐 Part 1:** RSA (Public Key Cryptography)

- Public key → shared
- Private key → secret
- Encrypt with public key, decrypt with private key


In [ ]:
# Generate PRIVATE key
# public_exponent=65537 is a standard safe value used in RSA
# key_size=2048 means 2048-bit key (secure for most purposes)
private_key = rsa.generate_private_key(
    public_exponent=65537,   # mathematical parameter used in RSA
    key_size=2048            # size of the key in bits (larger = more secure but slower)
)

# Generate PUBLIC key from private key
# In RSA, both keys are mathematically linked
public_key = private_key.public_key()

print("Keys generated successfully")

In [ ]:
import base64

# Define message to encrypt
# .encode() converts string → bytes (required for crypto functions)
message = "Hello secure world".encode()

# Encrypt using PUBLIC key
encrypted = public_key.encrypt(
    message,  # data to encrypt (must be bytes)
    padding.OAEP(  # OAEP = Optimal Asymmetric Encryption Padding (secure padding)
        mgf=padding.MGF1(algorithm=hashes.SHA256()),  # MGF1 = Mask Generation Function
        algorithm=hashes.SHA256(),  # hash algorithm used internally
        label=None  # optional label (not needed here)
    )
)

print("Encrypted (Python bytes object):", encrypted) # This will show as a bytes object (not human-readable). ASCIIs are not suitable for binary data, so we will convert it to hex and base64 for display.
print("Encrypted (hex):", encrypted.hex())
print("Encrypted (base64):", base64.b64encode(encrypted).decode()) # Standard way to represent binary data as text

In [ ]:
# Decrypt using PRIVATE key
decrypted = private_key.decrypt(
    encrypted,  # encrypted data
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Convert bytes → string using .decode()
print("Decrypted:", decrypted.decode())

When you:

1. Take a message  
2. Encrypt it with the **receiver’s public key**  
3. Encode it in **Base64**

You already achieved **confidentiality**:  
Only the holder of the **private key** can decrypt it.

> ⚠️ **Base64 is NOT security**, it is just an encoding format to represent binary data as text.

The real problem (without certificates)

The key question is:

> ❗ How do you know that the **public key** actually belongs to the right person?

Without certificates, this can happen:

- An attacker gives you **their public key**
- You encrypt the message using that key
- The attacker decrypts it with their private key
- Then re-encrypts it using the real recipient’s key

*Result:* your communication was intercepted without you noticing.

This is known as a **Man-in-the-Middle** attack.

So… what are certificates for?

A certificate (e.g., `X.509`) is used to:

**Bind:**
- A **public key**
- To a **real identity** (person, server, organization)

**It also includes:**
- A trusted signature (from a Certificate Authority)
- Owner information
- Expiration/validity dates

Who guarantees this?

A **Certificate Authority (CA)** such as:

- Let's Encrypt  
- DigiCert  

These entities sign certificates and essentially say:

> “Yes, this public key belongs to this entity.”

## **✍️ Part 2:** Digital Signatures

- Sign with PRIVATE key
- Verify with PUBLIC key


In [ ]:
# Create digital signature
signature = private_key.sign(
    message,  # original message
    padding.PSS(  # Probabilistic Signature Scheme (secure signature padding)
        mgf=padding.MGF1(hashes.SHA256()),  # mask generation function
        salt_length=padding.PSS.MAX_LENGTH  # random salt for security
    ),
    hashes.SHA256()  # hash algorithm used before signing
)

print("Signature created")

In [ ]:
# Verify signature
try:
    public_key.verify(
        signature,  # signature to verify
        message,    # original message
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Signature VALID")
except:
    print("Signature INVALID")

How it fits together?

In protocols like **Transport Layer Security (TLS)**:

1. The server sends its certificate  
2. The client verifies:
   - The CA signature  
   - The domain name  
3. **Only then** the public key is trusted  
4. The key is used for encryption

## **🌐 Part 3:** TLS Handshake Simulation

Real TLS:
1. Client gets server public key
2. Client creates session key
3. Client encrypts session key with public key
4. Server decrypts
5. Both use symmetric encryption


In [ ]:
# Import symmetric encryption tool
from cryptography.fernet import Fernet

In [ ]:
# Client generates symmetric session key
# This key will be used for fast encryption (like AES)
session_key = Fernet.generate_key()

# Create cipher object for encryption/decryption
cipher_client = Fernet(session_key)

print("Session key created")

In [ ]:
# Client encrypts session key using server PUBLIC key
encrypted_session_key = public_key.encrypt(
    session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

In [ ]:
# Server decrypts session key using PRIVATE key
decrypted_session_key = private_key.decrypt(
    encrypted_session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Server creates cipher with shared key
cipher_server = Fernet(decrypted_session_key)


In [ ]:
# Secure communication
secure_message = cipher_client.encrypt(b"Hello TLS secure world")

print("Encrypted message:", secure_message)

print("Decrypted message:", cipher_server.decrypt(secure_message).decode())

## **🧨 Part 4:** Mini CTF (Attacks)

In [ ]:
# Attack 1: Tampering
tampered = b"Hacked message"

try:
    cipher_server.decrypt(tampered)
    print("Unexpected success")
except:
    print("Tampering detected")

In [ ]:
# Attack 2: Wrong key
fake_key = Fernet.generate_key()
fake_cipher = Fernet(fake_key)

try:
    fake_cipher.decrypt(secure_message)
    print("Unexpected success")
except:
    print("Attack failed: wrong key")

In [ ]:
# Attack 3: Signature forgery
fake_message = b"Fake"

try:
    public_key.verify(
        signature,
        fake_message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Forgery success")
except:
    print("Forgery detected")

## **Assignment:** Secure Communication Challenge

### **Objective**
Apply all concepts from the lab:
- Confidentiality (encryption)
- Authentication (digital signatures)
- Integrity (verification)
- Secure communication (TLS simulation)
- Attack thinking (CTF)

### **Group Dynamics (Balanced Work)**

Each group must complete ALL roles:

1. **Sender**
2. **Receiver**
3. **Attacker**

You will interact with at least **two other groups**.

### **Task 1:** Secure Message Sending

Each group must:

1. Generate an RSA key pair
2. Share ONLY the **public key**
3. Create a message
4. Encrypt the message using the receiver’s public key
5. Sign the message using your private key

👉 Send to another group:
- Encrypted message
- Signature
- Your public key

### **Task 2:** Secure Message Receiving

Each group must:

1. Receive:
   - Encrypted message
   - Signature
   - Sender's public key

2. Decrypt the message using your private key  
3. Verify the signature using the sender’s public key  

✔ If valid → accept message  
❌ If invalid → reject message  

### **Task 3:** TLS Simulation

Repeat the communication using:

1. Session key (Fernet)
2. Encrypt session key with RSA
3. Exchange secure message

👉 Explain briefly:
- Why TLS uses BOTH asymmetric and symmetric encryption

### **Task 4:** CTF (Attack Phase)

Each group must attempt at least ONE attack:

- Modify a message (tampering)
- Try decrypting with wrong key
- Try breaking signature verification

👉 Report:
- What you tried
- What happened
- Why it failed (or succeeded)

### **Time Limit**
- 20–25 minutes total

### **Evaluation (Checklist)**

| Criteria | ✔ / ✘ |
|--------|------|
| RSA encryption implemented | ✔ / ✘ |
| Digital signature implemented | ✔ / ✘ |
| TLS simulation completed | ✔ / ✘ |
| Correctly decrypted & verified message | ✔ / ✘ |
| Performed at least one attack | ✔ / ✘ |
| Explanation of attack outcome | ✔ / ✘ |

### **Reflection Questions**

1. Why is encryption alone NOT enough for security?  
2. What is the role of digital signatures?  
3. Why does TLS use a hybrid approach?  
4. What was the easiest attack? Why did it fail?  

### **Bonus (Optional)**

- Save encrypted messages into files and exchange them
- Try to simulate a Man-in-the-Middle (MITM) attack
- Measure time difference: RSA vs symmetric encryption